# RAG 深入演示：Milvus Hybrid Search + 引用回答

这份 Notebook 面向现场分享，只保留一条可演示的完整链路：

```text
制度文档 -> 层级切分 -> Dense Embedding -> Milvus
                                  ├─ Dense 语义检索
                                  ├─ BM25 关键词检索
                                  └─ RRF Hybrid Search
                                           -> 带引用回答 -> 无证据拒答
```

> 安全约定：每次演示创建一个带时间戳的新 Collection，不删除、不清空、不覆盖已有数据。
> 请从项目根目录启动 Jupyter，再按顺序运行单元。

## 1. 环境预检

这个单元只检查配置、依赖、示例文档和 Milvus 端口，不创建或查询 Collection。

In [1]:
import os
import socket
from pathlib import Path
from urllib.parse import urlparse

from dotenv import load_dotenv


def find_project_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "pyproject.toml").exists() and (candidate / ".env").exists():
            return candidate
    raise FileNotFoundError("没有找到同时包含 pyproject.toml 和 .env 的项目根目录。")


PROJECT_ROOT = find_project_root()
load_dotenv(PROJECT_ROOT / ".env")

MILVUS_URI = os.getenv("MILVUS_URI", "http://127.0.0.1:19530")
MILVUS_TOKEN = os.getenv("MILVUS_TOKEN")
DENSE_DIM = int(os.getenv("DASHSCOPE_EMBEDDING_DIMENSIONS", "1024"))
EMBEDDING_MODEL_NAME = os.getenv("DASHSCOPE_EMBEDDING_MODEL", "text-embedding-v4")
EMBEDDING_BASE_URL = os.getenv(
    "DASHSCOPE_EMBEDDING_BASE_URL",
    "https://dashscope.aliyuncs.com/compatible-mode/v1",
)
EMBEDDING_BATCH_SIZE = min(10, max(1, int(os.getenv("DASHSCOPE_EMBEDDING_BATCH_SIZE", "10"))))
SOURCE_PATH = PROJECT_ROOT / "docs" / "sample_docs" / "enterprise_reimbursement_policy.md"

required_modules = [
    "pymilvus",
    "langchain_openai",
    "dotenv",
    "IPython",
]
module_status = {}
for module_name in required_modules:
    try:
        __import__(module_name)
        module_status[module_name] = "OK"
    except Exception as exc:
        module_status[module_name] = f"FAIL: {type(exc).__name__}: {exc}"

parsed_uri = urlparse(MILVUS_URI)
milvus_host = parsed_uri.hostname or "127.0.0.1"
milvus_port = parsed_uri.port or 19530
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
    sock.settimeout(1.5)
    milvus_port_open = sock.connect_ex((milvus_host, milvus_port)) == 0

print("项目根目录：", PROJECT_ROOT)
print("示例文档：", SOURCE_PATH, "存在=", SOURCE_PATH.exists())
print("Milvus URI：", MILVUS_URI, "端口可连接=", milvus_port_open)
print("Embedding：", EMBEDDING_MODEL_NAME, "dim=", DENSE_DIM, "batch=", EMBEDDING_BATCH_SIZE)
print("DASHSCOPE_API_KEY：", "已配置" if os.getenv("DASHSCOPE_API_KEY") else "缺失")
print("DEEPSEEK_API_KEY：", "已配置" if os.getenv("DEEPSEEK_API_KEY") else "缺失")
print("依赖状态：", module_status)

if not SOURCE_PATH.exists():
    raise FileNotFoundError(SOURCE_PATH)
if not milvus_port_open:
    print("提示：请先启动 Docker Desktop 和本地 Milvus，再运行创建 Collection 的单元。")

项目根目录： D:\PythonProject\LearnOne
示例文档： D:\PythonProject\LearnOne\docs\sample_docs\enterprise_reimbursement_policy.md 存在= True
Milvus URI： http://127.0.0.1:19530 端口可连接= False
Embedding： text-embedding-v4 dim= 2048 batch= 10
DASHSCOPE_API_KEY： 已配置
DEEPSEEK_API_KEY： 已配置
依赖状态： {'pymilvus': 'OK', 'langchain_openai': 'OK', 'dotenv': 'OK', 'IPython': 'OK'}
提示：请先启动 Docker Desktop 和本地 Milvus，再运行创建 Collection 的单元。


## 2. 层级切分：先保留标题，再按长度切 Chunk

演示重点：

- `section_path` 保留文档层级。
- `parent_id` 支持命中子块后扩展父级或邻近上下文。
- `overlap` 降低规则在边界处被切断的概率。
- Dense Embedding 使用标题路径和邻近内容；BM25 使用原始正文。

In [ ]:
import re
import uuid
from dataclasses import asdict, dataclass
from pprint import pprint


@dataclass
class Section:
    doc_id: str
    section_id: str
    source: str
    heading: str
    section_path: list[str]
    text: str


@dataclass
class Chunk:
    chunk_id: str
    parent_id: str
    doc_id: str
    source: str
    section_path: list[str]
    chunk_index: int
    text: str
    embedding_text: str


def parse_markdown_sections(markdown: str, source: str, doc_id: str) -> list[Section]:
    heading_pattern = re.compile(r"^(#{1,6})\s+(.+?)\s*$")
    stack: list[tuple[int, str]] = []
    sections: list[Section] = []
    current_heading = "文档开头"
    current_path: list[str] = []
    current_lines: list[str] = []

    def flush() -> None:
        text = "\n".join(current_lines).strip()
        if not text:
            return
        section_path = list(current_path or [current_heading])
        section_key = " > ".join(section_path)
        sections.append(
            Section(
                doc_id=doc_id,
                section_id=str(uuid.uuid5(uuid.NAMESPACE_URL, f"{doc_id}:{section_key}")),
                source=source,
                heading=current_heading,
                section_path=section_path,
                text=text,
            )
        )

    for raw_line in markdown.splitlines():
        match = heading_pattern.match(raw_line)
        if match:
            flush()
            level = len(match.group(1))
            current_heading = match.group(2).strip()
            stack = [(item_level, title) for item_level, title in stack if item_level < level]
            stack.append((level, current_heading))
            current_path = [title for _, title in stack]
            current_lines = []
        else:
            current_lines.append(raw_line)
    flush()
    return sections


def split_with_overlap(text: str, chunk_size: int = 260, overlap: int = 60) -> list[str]:
    clean_text = re.sub(r"\n{3,}", "\n\n", text).strip()
    if len(clean_text) <= chunk_size:
        return [clean_text]

    pieces = []
    start = 0
    while start < len(clean_text):
        end = min(start + chunk_size, len(clean_text))
        boundaries = [
            clean_text.rfind("\n\n", start, end),
            clean_text.rfind("。", start, end),
            clean_text.rfind("；", start, end),
        ]
        boundary = max(boundaries)
        if boundary > start + int(chunk_size * 0.55):
            end = boundary + 1
        pieces.append(clean_text[start:end].strip())
        if end >= len(clean_text):
            break
        start = max(0, end - overlap)
    return pieces


def build_chunks(sections: list[Section]) -> list[Chunk]:
    chunks: list[Chunk] = []
    for section in sections:
        pieces = split_with_overlap(section.text)
        for index, piece in enumerate(pieces):
            previous_tail = pieces[index - 1][-80:] if index > 0 else ""
            next_head = pieces[index + 1][:80] if index < len(pieces) - 1 else ""
            path_text = " > ".join(section.section_path)
            embedding_text = (
                f"标题路径：{path_text}\n"
                f"上一段尾部：{previous_tail or '无'}\n"
                f"当前片段：{piece}\n"
                f"下一段开头：{next_head or '无'}"
            )
            chunks.append(
                Chunk(
                    chunk_id=str(uuid.uuid5(uuid.NAMESPACE_URL, f"{section.section_id}:{index}:{piece}")),
                    parent_id=section.section_id,
                    doc_id=section.doc_id,
                    source=section.source,
                    section_path=section.section_path,
                    chunk_index=index,
                    text=piece,
                    embedding_text=embedding_text,
                )
            )
    return chunks


policy_markdown = SOURCE_PATH.read_text(encoding="utf-8")
sections = parse_markdown_sections(
    policy_markdown,
    source=SOURCE_PATH.name,
    doc_id="policy-demo-001",
)
chunks = build_chunks(sections)

print(f"Section 数量：{len(sections)}")
print(f"Chunk 数量：{len(chunks)}")
print("\n第一个 Chunk：")
pprint(asdict(chunks[0]), width=120)

## 3. 生成 Dense Embedding

Dense 向量负责语义匹配；BM25 Sparse 向量由 Milvus 根据 `text` 自动生成。

> 本单元会调用 `.env` 中配置的百炼 Embedding 服务。

In [ ]:
from langchain_openai import OpenAIEmbeddings

dashscope_api_key = os.getenv("DASHSCOPE_API_KEY")
if not dashscope_api_key:
    raise RuntimeError(".env 中缺少 DASHSCOPE_API_KEY。")

embedding_model = OpenAIEmbeddings(
    model=EMBEDDING_MODEL_NAME,
    api_key=dashscope_api_key,
    base_url=EMBEDDING_BASE_URL,
    dimensions=DENSE_DIM,
    check_embedding_ctx_length=False,
    chunk_size=EMBEDDING_BATCH_SIZE,
)

dense_vectors = embedding_model.embed_documents([chunk.embedding_text for chunk in chunks])
print("向量数量：", len(dense_vectors))
print("向量维度：", len(dense_vectors[0]))
print("首个向量前 8 维：", [round(value, 5) for value in dense_vectors[0][:8]])

## 4. 创建 Milvus Dense + BM25 Collection

这个单元会执行以下本地 Milvus 数据操作：

1. 创建一个新的时间戳 Collection。
2. 创建 Dense 和 Sparse 索引。
3. 插入本次演示的 24 个左右 Chunk。
4. Flush 并 Load 新 Collection。

不会删除、清空、覆盖或重建任何已有 Collection。

In [ ]:
from datetime import datetime

from pymilvus import DataType, Function, FunctionType, MilvusClient

client_kwargs = {"uri": MILVUS_URI, "timeout": 30}
if MILVUS_TOKEN:
    client_kwargs["token"] = MILVUS_TOKEN
client = MilvusClient(**client_kwargs)

COLLECTION_NAME = f"learnone_rag_demo_{datetime.now():%Y%m%d_%H%M%S}"

schema = client.create_schema(auto_id=False, enable_dynamic_field=False)
schema.add_field("chunk_id", DataType.VARCHAR, is_primary=True, max_length=80)
schema.add_field("parent_id", DataType.VARCHAR, max_length=80)
schema.add_field("doc_id", DataType.VARCHAR, max_length=80)
schema.add_field("source", DataType.VARCHAR, max_length=256)
schema.add_field("section_path", DataType.VARCHAR, max_length=512)
schema.add_field("chunk_index", DataType.INT64)
schema.add_field("text", DataType.VARCHAR, max_length=4096, enable_analyzer=True)
schema.add_field("embedding_text", DataType.VARCHAR, max_length=8192)
schema.add_field("dense_vector", DataType.FLOAT_VECTOR, dim=DENSE_DIM)
schema.add_field("sparse_vector", DataType.SPARSE_FLOAT_VECTOR)
schema.add_function(
    Function(
        name="text_bm25_emb",
        input_field_names=["text"],
        output_field_names=["sparse_vector"],
        function_type=FunctionType.BM25,
    )
)

index_params = client.prepare_index_params()
index_params.add_index(
    field_name="dense_vector",
    index_name="dense_vector_index",
    index_type="FLAT",
    metric_type="COSINE",
)
index_params.add_index(
    field_name="sparse_vector",
    index_name="sparse_vector_index",
    index_type="SPARSE_INVERTED_INDEX",
    metric_type="BM25",
    params={"inverted_index_algo": "DAAT_MAXSCORE"},
)

client.create_collection(
    collection_name=COLLECTION_NAME,
    schema=schema,
    index_params=index_params,
)

rows = []
for chunk, dense_vector in zip(chunks, dense_vectors):
    rows.append(
        {
            "chunk_id": chunk.chunk_id,
            "parent_id": chunk.parent_id,
            "doc_id": chunk.doc_id,
            "source": chunk.source,
            "section_path": " > ".join(chunk.section_path),
            "chunk_index": chunk.chunk_index,
            "text": chunk.text,
            "embedding_text": chunk.embedding_text,
            "dense_vector": dense_vector,
        }
    )

insert_result = client.insert(collection_name=COLLECTION_NAME, data=rows)
client.flush(collection_name=COLLECTION_NAME)
client.load_collection(collection_name=COLLECTION_NAME)

print("Collection：", COLLECTION_NAME)
print("插入数量：", insert_result["insert_count"])
print("Milvus WebUI：http://127.0.0.1:9091/webui/")

## 5. Dense、BM25、Hybrid Search 对比

同一个问题同时走三条检索路线：

- Dense：擅长语义相似。
- BM25：擅长金额、编号、专有词等精确匹配。
- Hybrid + RRF：融合两路排名，不要求两种分数处于同一尺度。

In [ ]:
from IPython.display import Markdown, display
from pymilvus import AnnSearchRequest, RRFRanker

OUTPUT_FIELDS = [
    "chunk_id",
    "parent_id",
    "doc_id",
    "source",
    "section_path",
    "chunk_index",
    "text",
]


def normalize_hits(results) -> list[dict]:
    normalized = []
    for hit in results[0]:
        entity = hit.get("entity", {})
        normalized.append(
            {
                **entity,
                "score": float(hit.get("distance", hit.get("score", 0.0))),
            }
        )
    return normalized


def search_three_routes(question: str, limit: int = 5) -> dict[str, list[dict]]:
    query_dense = embedding_model.embed_query(question)
    filter_expr = 'doc_id == "policy-demo-001"'

    dense_results = client.search(
        collection_name=COLLECTION_NAME,
        data=[query_dense],
        anns_field="dense_vector",
        search_params={"metric_type": "COSINE", "params": {}},
        filter=filter_expr,
        limit=limit,
        output_fields=OUTPUT_FIELDS,
    )
    sparse_results = client.search(
        collection_name=COLLECTION_NAME,
        data=[question],
        anns_field="sparse_vector",
        search_params={"metric_type": "BM25", "params": {}},
        filter=filter_expr,
        limit=limit,
        output_fields=OUTPUT_FIELDS,
    )

    dense_request = AnnSearchRequest(
        data=[query_dense],
        anns_field="dense_vector",
        param={"metric_type": "COSINE", "params": {}},
        limit=limit,
        expr=filter_expr,
    )
    sparse_request = AnnSearchRequest(
        data=[question],
        anns_field="sparse_vector",
        param={"metric_type": "BM25", "params": {}},
        limit=limit,
        expr=filter_expr,
    )
    hybrid_results = client.hybrid_search(
        collection_name=COLLECTION_NAME,
        reqs=[dense_request, sparse_request],
        ranker=RRFRanker(k=60),
        limit=limit,
        output_fields=OUTPUT_FIELDS,
    )
    return {
        "Dense": normalize_hits(dense_results),
        "BM25": normalize_hits(sparse_results),
        "Hybrid RRF": normalize_hits(hybrid_results),
    }


def render_search_results(title: str, hits: list[dict]) -> None:
    lines = [
        f"### {title}",
        "",
        "| # | score | section | text preview |",
        "| --- | ---: | --- | --- |",
    ]
    for index, hit in enumerate(hits, start=1):
        section = hit.get("section_path", "").replace("|", "&#124;")
        preview = hit.get("text", "").replace("\n", " ").replace("|", "&#124;")[:100]
        lines.append(f"| {index} | {hit['score']:.4f} | {section} | {preview} |")
    display(Markdown("\n".join(lines)))


QUESTION = "报销超过 5000 元的差旅费，需要经过哪些审批？"
SEARCH_RESULTS = search_three_routes(QUESTION)

print("问题：", QUESTION)
for route_name, route_hits in SEARCH_RESULTS.items():
    render_search_results(route_name, route_hits)

## 6. Hybrid 结果生成带引用回答

检索结果先格式化成编号上下文，再要求模型只能基于证据回答，并使用 `[1]`、`[2]` 标注来源。

In [ ]:
from langchain_openai import ChatOpenAI


def build_context(hits: list[dict], top_k: int = 4) -> str:
    blocks = []
    for index, hit in enumerate(hits[:top_k], start=1):
        blocks.append(
            f"[{index}] source={hit['source']} | section={hit['section_path']} | chunk={hit['chunk_index']}\n"
            f"{hit['text']}"
        )
    return "\n\n".join(blocks)


def build_chat_model() -> ChatOpenAI:
    api_key = os.getenv("DEEPSEEK_API_KEY")
    if not api_key:
        raise RuntimeError(".env 中缺少 DEEPSEEK_API_KEY。")
    return ChatOpenAI(
        model=os.getenv("DEEPSEEK_MODEL") or os.getenv("OPENAI_MODEL") or "deepseek-chat",
        api_key=api_key,
        base_url=os.getenv("DEEPSEEK_BASE_URL", "https://api.deepseek.com"),
        temperature=0,
        timeout=float(os.getenv("DEEPSEEK_TIMEOUT", "120")),
        max_retries=int(os.getenv("DEEPSEEK_MAX_RETRIES", "2")),
    )


def answer_with_citations(question: str, hits: list[dict]) -> dict:
    context = build_context(hits)
    prompt = f"""你是企业制度问答助手。请严格遵守：
1. 只能根据给定上下文回答。
2. 如果上下文没有答案，明确回答“不知道，根据当前制度资料无法确认”。
3. 每个关键结论必须使用 [1]、[2] 形式标注来源。
4. 不得补充上下文中没有出现的审批人、金额或例外规则。

上下文：
{context}

问题：{question}
答案："""
    response = build_chat_model().invoke(prompt)
    return {"question": question, "context": context, "answer": response.content}


KNOWN_ANSWER = answer_with_citations(QUESTION, SEARCH_RESULTS["Hybrid RRF"])
display(Markdown(f"### 最终回答\n\n{KNOWN_ANSWER['answer']}"))
display(Markdown(f"### 实际发送给模型的证据\n\n```text\n{KNOWN_ANSWER['context']}\n```"))

## 7. 无证据拒答

企业 RAG 的重要能力不是每次都回答，而是在知识库没有证据时稳定拒答。

下面的问题包含“宠物”和“医疗”两个关键概念；如果召回内容不包含这些证据，程序直接拒答，不把无关上下文交给模型猜测。

In [ ]:
def answer_with_evidence_gate(
    question: str,
    required_terms: list[str],
) -> dict:
    search_results = search_three_routes(question, limit=5)
    hybrid_hits = search_results["Hybrid RRF"]
    evidence_text = "\n".join(hit.get("text", "") for hit in hybrid_hits)
    missing_terms = [term for term in required_terms if term not in evidence_text]

    if missing_terms:
        return {
            "question": question,
            "answer": "不知道，根据当前制度资料无法确认。",
            "reason": f"召回证据缺少关键概念：{', '.join(missing_terms)}",
            "llm_called": False,
        }

    result = answer_with_citations(question, hybrid_hits)
    return {**result, "reason": "证据检查通过", "llm_called": True}


UNKNOWN_QUESTION = "公司的宠物医疗费应该如何报销？"
REFUSAL_RESULT = answer_with_evidence_gate(UNKNOWN_QUESTION, required_terms=["宠物", "医疗"])

print("问题：", REFUSAL_RESULT["question"])
print("回答：", REFUSAL_RESULT["answer"])
print("原因：", REFUSAL_RESULT["reason"])
print("是否调用 LLM：", REFUSAL_RESULT["llm_called"])